# 📘 부록: 로컬 LLM Gemma 4로 파인튜닝 데이터셋 만들기

교안 01·02는 OpenAI API의 `gpt-5.6-luna`를 생성용 LLM으로 씁니다. 이 부록은 같은 본실습 데이터셋을 GPU 파드의 로컬 LLM(Ollama의 Gemma 4 E4B)으로 만들고, 그 전에 API와 로컬 LLM 다섯 개를 같은 표본으로 비교한 측정 결과를 정리합니다. 로컬 LLM은 회사 밖으로 보낼 수 없는 문서로 학습 데이터를 만들 때 씁니다.

**이 부록의 흐름**
1. 생성용 LLM 비교: 측정 조건, 속도, 비용, 품질 판정 방법과 결과
2. 준비: RunPod 파드에 Ollama 설치, 교안 01과 같은 함수와 체인
3. 본실습 데이터셋을 Gemma 4 E4B로 만들기: 단계마다 진행 파일에 이어 쓰고, 멈춘 곳부터 다시 실행
4. 교안 02에서 이 파일 쓰기
5. 핵심 정리

생성용 LLM 비교는 측정 기록이라 읽기만 합니다. 준비와 데이터셋 만들기의 코드 셀은 GPU 파드에서 Ollama 서버를 띄운 뒤 실행합니다.

## 1. 생성용 LLM 비교: API와 로컬 LLM

교안 01·02의 생성 체인을 API(`gpt-5.6-luna`)와 로컬 LLM 다섯 개로 같은 표본에 보내 속도, 비용, 품질을 측정했습니다. 숫자는 모두 설정마다 한 번 측정한 값이라, 다시 측정하면 조금씩 달라집니다.

### 1-1. 측정 조건

**측정 환경**

| 항목 | 조건 |
|---|---|
| GPU | RunPod RTX PRO 4000(24GB) 한 장, 시간당 0.59달러 |
| 로컬 서버 | Ollama 0.40.11, `ChatOllama(model=태그, reasoning=False)` |
| 양자화 | 로컬 모델 다섯 개 모두 q4_K_M(가중치를 약 4비트로 줄여 저장한 판) |
| API | `gpt-5.6-luna`, 동시 요청 32 |

**비교한 로컬 모델**

| 생성용 LLM | Ollama 태그 | 파라미터 수 |
|---|---|---|
| Gemma 4 E4B (q4_K_M 양자화) | `gemma4:e4b-it-q4_K_M` | 7.5B |
| Gemma 4 12B (q4_K_M 양자화) | `gemma4:12b-it-q4_K_M` | 11.9B |
| Gemma 4 26B MoE (q4_K_M 양자화) | `gemma4:26b-a4b-it-q4_K_M` | 25.8B(MoE, 토큰마다 약 4B만 계산) |
| Qwen3.5 4B (q4_K_M 양자화) | `qwen3.5:4b-q4_K_M` | 4.2B |
| Qwen3.5 9B (q4_K_M 양자화) | `qwen3.5:9b-q4_K_M` | 9.0B |

**표본** (여섯 생성용 LLM에 똑같은 입력과 프롬프트를 넣었습니다)
- **Q&A 생성 40문단:** 노동법 생성 대상 382문단에서 주제 태그마다 골라 모은 40문단이고, 문단마다 Q&A를 1~3개 요청했습니다.
- **패러프레이즈 40개:** 전월세 따라하기 데이터의 질문 40개이고, 빈 질문 1개를 빼고 39개를 판정했습니다.
- **새 질문에 답 만들기 3건:** 기존 FAQ의 임금 질문 3개입니다. 측정 당시에는 임금 주제의 조문 전체를 함께 넣었고, 지금 교안은 검색한 조문 5개를 넣습니다.

### 1-2. 속도와 동시 처리

생성용 LLM마다 서버의 동시 처리 수(`OLLAMA_NUM_PARALLEL`)를 바꿔 가며 Q&A 생성 40건과 패러프레이즈 40건을 보내고, 가장 빠른 설정만 모았습니다. 처리 속도는 1초에 끝낸 요청 수(보낸 요청 수 / 걸린 시간)이고, Q&A 생성은 요청 하나가 조문 문단 하나, 패러프레이즈는 요청 하나가 질문 하나입니다. 예를 들어 Q&A 생성 1.10은 1초에 문단 약 1.1개를 처리한다는 뜻입니다.

| 생성용 LLM | 가장 빠른 동시 처리 수 | Q&A 생성 속도(초당 문단 수) | 패러프레이즈 속도(초당 질문 수) | GPU 최대 메모리(MiB) |
|---|---|---|---|---|
| gpt-5.6-luna (API) | 32 | 3.57 | 10.11 | - |
| Gemma 4 E4B (q4_K_M 양자화) | 32 | 1.10 | 4.00 | 8,162 |
| Gemma 4 12B (q4_K_M 양자화) | 8 | 0.86 | 1.75 | 14,124 |
| Gemma 4 26B MoE (q4_K_M 양자화) | 4 | 0.87 | 3.12 | 19,852 |
| Qwen3.5 4B (q4_K_M 양자화) | 1 (동시 처리 안 됨) | 0.29 | 2.07 | 3,924 |
| Qwen3.5 9B (q4_K_M 양자화) | 1 (동시 처리 안 됨) | 0.22 | 1.57 | 6,512 |

**결과 읽기**
- **Gemma 4 E4B와 12B:** E4B는 동시 32가 가장 빠르고 GPU 메모리는 약 8GB(8,162MiB)입니다. 12B는 동시 16에서 메모리가 24GB를 거의 다 차서 느려졌고, 동시 32에서는 패러프레이즈 40건 가운데 32건이 실패했습니다.
- **Qwen3.5:** 서버 기록에 `model architecture does not currently support parallel requests` 경고가 나와, 이 Ollama 판은 요청을 하나씩 처리해 동시 처리 수를 올려도 빨라지지 않습니다.
- **`.batch`와 서버:** `max_concurrency`는 요청을 동시에 몇 개까지 보낼지만 정하고, 실제 동시 처리 수는 서버(Ollama 설정, 모델 구조, GPU 메모리)가 정합니다.

### 1-3. 비용과 시간 어림

학생 한 명이 교안 01·02를 처음부터 끝까지 한 번 실행하는 규모로 어림했습니다.

| 계열 | 호출 수 | 근거 |
|---|---|---|
| 생성 | 약 440 | 생성 대상 382문단 + 함께 따라하기 45문단 + 질문 늘리기 12회 + 범위 밖 1회 |
| 새 질문에 답 만들기 | 약 200 | 질문 늘리기에서 남은 지시마다 1회 |
| 증강 | 약 1,100 | 패러프레이즈 약 700 + 역번역 400 |

- **API 비용:** 호출 수 x 호출당 평균 입력·출력 토큰 수 x 단가(100만 토큰당 입력 0.20달러, 출력 1.20달러, [OpenAI API 가격](https://developers.openai.com/api/docs/pricing))로 계산했습니다.
- **로컬 비용:** 실행 시간(호출 수 / 처리 속도)과 준비 시간(파드 시작, 설치, 모델 내려받기)에 시간당 요금을 곱했습니다.

| 생성용 LLM | 생성 440회 | 답 만들기 200회 | 증강 1,100회 | 준비 | 합계 시간 | 비용 |
|---|---|---|---|---|---|---|
| gpt-5.6-luna (API) | 78.5초, 0.229달러 | 22.3초, 0.291달러 | 69.1초, 0.117달러 | 없음 | 2.8분 | 0.638달러 |
| Gemma 4 E4B (q4_K_M 양자화) | 6.6분(동시 32, 초당 1.103개) | 6.2분(동시 4, 초당 0.541개) | 4.6분(동시 32, 초당 3.996개) | 119초 | 19.4분 | 0.191달러 |
| Gemma 4 12B (q4_K_M 양자화) | 8.6분(동시 8, 초당 0.857개) | 12.0분(동시 4, 초당 0.277개) | 10.5분(동시 8, 초당 1.751개) | 118초 | 33.0분 | 0.325달러 |
| Gemma 4 26B MoE (q4_K_M 양자화) | 8.5분(동시 4, 초당 0.866개) | 6.1분(동시 2, 초당 0.550개) | 5.9분(동시 4, 초당 3.118개) | 159초 | 23.1분 | 0.227달러 |
| Qwen3.5 4B (q4_K_M 양자화) | 25.3분(동시 1, 초당 0.290개) | 5.1분(동시 1, 초당 0.658개) | 8.9분(동시 1, 초당 2.071개) | 108초 | 41.0분 | 0.403달러 |
| Qwen3.5 9B (q4_K_M 양자화) | 32.9분(동시 1, 초당 0.223개) | 6.6분(동시 1, 초당 0.507개) | 11.7분(동시 1, 초당 1.566개) | 115초 | 53.1분 | 0.522달러 |

**결과 읽기**
- **API와 로컬:** API가 가장 빠르고(2.8분), 로컬 가운데는 Gemma 4 E4B가 가장 빠르고 쌉니다(19.4분, 0.191달러).
- **답 만들기의 비용 비중:** API에서 새 질문에 답 만들기는 입력에 주제 태그의 조문 전체가 들어가, 비용(0.291달러)이 생성(0.229달러)보다 크고 전체의 약 46%입니다. 지금 교안은 검색한 조문 5개만 넣어 이 비용이 줄어듭니다.

### 1-4. 품질 판정 방법과 지표 정의

**판정 방법:** 생성에 쓰지 않은 LLM 검토자 한 명이 생성용 LLM 이름을 가린 출력(블라인드)을 판정했습니다. 대상은 여섯 생성용 LLM이 40문단에서 만든 Q&A 502건입니다.

Q&A 한 건마다 그 Q&A를 만든 근거 문단 하나와 대조해 아래 다섯 범주 가운데 하나로 판정했습니다.

| 범주 | 정의 |
|---|---|
| 근거 있음 정확 | 문단 내용만으로 맞고, 문단의 조건과 예외를 지킨 답 |
| 조문에 없는 내용 추가 | 문단에 없는 사실, 절차, 조언, 단정을 덧붙인 답 |
| 숫자·조건 바뀜 | 숫자, 대상, 권리와 의무, 주체, 결론이 문단과 달라진 답 |
| 조건·예외 누락 | 규칙은 맞게 말했지만 문단의 조건이나 예외가 빠져 오해를 부르는 답 |
| 질문과 안 맞음 | 질문이 문단 밖이거나, 답이 질문에 답하지 않음 |

**지표 정의**

모든 비율의 분모는 그 생성용 LLM이 40문단에서 만든 Q&A 전체 건수입니다. 다섯 범주는 서로 겹치지 않아 비율의 합이 100%입니다.

- **정확** = 근거 있음 정확으로 판정한 건수 / Q&A 전체 건수
- **할루시네이션**(hallucination: 근거에 없는 내용을 사실처럼 쓰는 오류) = (조문에 없는 내용 추가 + 숫자·조건 바뀜) 건수 / Q&A 전체 건수
- **조건·예외 누락** = 조건·예외 누락으로 판정한 건수 / Q&A 전체 건수
- **질문과 안 맞음** = 질문과 안 맞음으로 판정한 건수 / Q&A 전체 건수
- **금지 표현** = 답에 `BANNED_PHRASES`의 앞 여섯 개 가운데 하나라도 있는 Q&A 건수 / Q&A 전체 건수(다섯 범주와 따로 셈)

| 생성용 LLM | Q&A 전체(분모) | 정확 | 조문에 없는 내용 추가 | 숫자·조건 바뀜 | 할루시네이션(추가 + 바뀜) | 조건·예외 누락 | 질문과 안 맞음 | 금지 표현 |
|---|---|---|---|---|---|---|---|---|
| gpt-5.6-luna | 79 | 76 (96.2%) | 1 | 0 | 1 (1.3%) | 2 (2.5%) | 0 (0.0%) | 0 (0.0%) |
| Gemma 4 E4B (q4_K_M 양자화) | 71 | 63 (88.7%) | 0 | 1 | 1 (1.4%) | 5 (7.0%) | 2 (2.8%) | 0 (0.0%) |
| Gemma 4 12B (q4_K_M 양자화) | 64 | 56 (87.5%) | 0 | 1 | 1 (1.6%) | 6 (9.4%) | 1 (1.6%) | 0 (0.0%) |
| Gemma 4 26B MoE (q4_K_M 양자화) | 64 | 50 (78.1%) | 2 | 3 | 5 (7.8%) | 7 (10.9%) | 2 (3.1%) | 0 (0.0%) |
| Qwen3.5 4B (q4_K_M 양자화) | 116 | 54 (46.6%) | 7 | 28 | 35 (30.2%) | 8 (6.9%) | 19 (16.4%) | 33 (28.4%) |
| Qwen3.5 9B (q4_K_M 양자화) | 108 | 46 (42.6%) | 12 | 16 | 28 (25.9%) | 11 (10.2%) | 23 (21.3%) | 46 (42.6%) |

- **분모가 다른 이유:** 40문단 모두 문단마다 Q&A를 1~3개(물을 것이 적으면 적게) 요청했는데, 몇 개를 만들지는 생성용 LLM마다 달랐습니다. 문단당 평균이 gpt-5.6-luna 약 2.0개, Gemma 4 1.6~1.8개, Qwen3.5 2.7~2.9개라, 비율(%)로 비교합니다.
- **교안 필터의 한계:** 교안 01의 숫자 대조(`numbers_match`)와 금지 표현 필터는 Gemma 4 세 모델의 할루시네이션을 하나도 걸러 내지 못했습니다. 그래서 로컬 LLM으로 만든 데이터는 표본 검토가 필요합니다.

**판정 기준** (이 부록이 정한 값이고, 교안 필터가 할루시네이션을 거의 걸러 내지 못해 할루시네이션을 먼저 봅니다)

| 판정 | 수치 기준 | 해당 생성용 LLM |
|---|---|---|
| 그대로 사용 | 할루시네이션 2% 미만, 정확 95% 이상 | gpt-5.6-luna |
| 표본 검토 뒤 사용 | 할루시네이션 2% 미만, 정확 85% 이상 | Gemma 4 E4B, Gemma 4 12B |
| 전수 검토 | 할루시네이션 2% 이상 10% 미만 | Gemma 4 26B MoE |
| 사용하지 않음 | 할루시네이션 10% 이상 | Qwen3.5 4B, Qwen3.5 9B |

- **한계:** LLM 검토자 한 명이 한 번 판정했고, 모델마다 Q&A 64~116건이라 1건이 약 1~1.6%포인트입니다.

### 1-5. 결론

| 생성용 LLM | 교안 1회 시간 / 비용 | 정확 | 할루시네이션 | 조건·예외 누락 | 판정 |
|---|---|---|---|---|---|
| gpt-5.6-luna (API) | 약 2.8분 / 약 0.64달러 | 96.2% | 1.3% | 2.5% | 그대로 사용 |
| Gemma 4 E4B (q4_K_M 양자화) | 약 19분 / 약 0.19달러 | 88.7% | 1.4% | 7.0% | 표본 검토 뒤 사용 |
| Gemma 4 12B (q4_K_M 양자화) | 약 33분 / 약 0.32달러 | 87.5% | 1.6% | 9.4% | 표본 검토 뒤 사용 |
| Gemma 4 26B MoE (q4_K_M 양자화) | 약 23분 / 약 0.23달러 | 78.1% | 7.8% | 10.9% | 전수 검토 |
| Qwen3.5 4B (q4_K_M 양자화) | 약 41분 / 약 0.40달러 | 46.6% | 30.2% | 6.9% | 사용하지 않음 |
| Qwen3.5 9B (q4_K_M 양자화) | 약 53분 / 약 0.52달러 | 42.6% | 25.9% | 10.2% | 사용하지 않음 |

- **교안은 API:** `gpt-5.6-luna`가 정확 96.2%, 할루시네이션 1.3%로 품질이 가장 좋고, 준비 없이 약 2.8분, 약 0.64달러입니다.
- **로컬 후보는 Gemma 4 E4B (q4_K_M 양자화):** 품질이 Gemma 4 12B와 비슷하고, 로컬 가운데 가장 빠르고 쌉니다(약 19분, 약 0.19달러, GPU 메모리 약 8GB). 약점은 사실 확인 질문 쏠림(81.7%)과 필터를 통과하는 조건·예외 누락(7.0%)이라, 아래 데이터셋 만들기는 근거 검수 뒤에 표본 검토를 넣습니다.
- **Qwen3.5는 뺌:** 할루시네이션이 25.9~30.2%, 금지 표현이 28.4~42.6%이고, 이번 Ollama 판에서는 동시 처리도 되지 않습니다.

## 2. 준비: RunPod 파드와 Ollama

### 2-1. Ollama 설치와 서버

오픈소스 LLM 로컬 실행 단원과 같은 방법으로 GPU 파드 터미널에서 Ollama를 설치하고, 이번 설정으로 서버를 띄웁니다.

**리눅스 셸 명령(파드 터미널 1)**

```bash
# Ollama 설치
apt-get update && apt-get install -y zstd
curl -fsSL https://ollama.com/install.sh | sh

# 동시 처리 32개, 문맥 4096토큰, 모델 유지 30분으로 서버 시작(이 터미널은 서버가 쓰므로 열어 둡니다)
OLLAMA_NUM_PARALLEL=32 OLLAMA_CONTEXT_LENGTH=4096 OLLAMA_KEEP_ALIVE=30m ollama serve
```

- `OLLAMA_NUM_PARALLEL`은 서버가 동시에 처리하는 요청 수로, 준비 셀의 `MAX_CONCURRENCY`(32)와 맞춥니다.
- `OLLAMA_CONTEXT_LENGTH`는 요청 하나의 문맥 상한, `OLLAMA_KEEP_ALIVE`는 마지막 요청 뒤 모델을 GPU에 올려 두는 시간입니다.
- `address already in use`가 나오면 이미 떠 있는 서버를 `pkill ollama`로 끄고 다시 시작합니다.

**리눅스 셸 명령(파드 터미널 2)**

```bash
# Gemma 4 E4B(q4_K_M 양자화) 내려받기(약 6.6GB)
ollama pull gemma4:e4b-it-q4_K_M
```

- **검색기:** 교안 01의 `law_retriever.py`를 그대로 쓰므로 단원 폴더 `.env`에 `OPENAI_API_KEY`가 있어야 합니다.

### 2-2. 준비 셀

이 부록은 단원 폴더에 있으므로 `material_dir`은 `"."`입니다. 생성용 LLM은 `ChatOllama`로 파드의 Ollama 서버에 요청을 보내고, 나머지 체인 코드는 교안 01과 같습니다.

- **`MAX_CONCURRENCY`(32):** `.batch`가 동시에 보내는 요청 수의 상한입니다. 서버의 `OLLAMA_NUM_PARALLEL`과 같은 값으로 두고, 32는 이번 측정에서 가장 빨랐던 값입니다.
- **`CHUNK_SIZE`(64):** `.batch` 한 번에 보내는 입력 수이고, 묶음이 끝날 때마다 결과를 진행 파일에 씁니다. `.batch`는 묶음의 모든 요청이 끝나야 돌아오므로 묶음이 작으면 서버가 쉬는 시간이 늘고, 크면 중간에 멈췄을 때 다시 보내는 요청이 늘어 `MAX_CONCURRENCY`의 두 배로 잡았습니다.
- **진행 파일:** 단계마다 `output/gemma4_progress/` 아래에 하나씩 둡니다.

In [ ]:
# 경로, 생성용 LLM, 동시 요청 수, 단계별 진행 파일 경로를 준비합니다
import json
import time
from pathlib import Path

import pandas as pd
from langchain_ollama import ChatOllama

material_dir = Path(".")  # data가 있는 단원 폴더. 이 부록은 단원 폴더에 있습니다
data_dir = material_dir / "data"
output_dir = material_dir / "output"  # 이 단원이 만든 데이터셋 파일을 저장하는 폴더
output_dir.mkdir(exist_ok=True)

# 생성용 LLM: 파드에서 띄운 Ollama 서버(기본 주소 http://localhost:11434)의 Gemma 4 E4B
# reasoning=False: 답 전에 추론하는 단계(thinking)를 끕니다. 위의 비교 측정과 같은 설정입니다
MODEL_NAME = "gemma4:e4b-it-q4_K_M"
llm = ChatOllama(model=MODEL_NAME, reasoning=False)

# .batch가 동시에 보내는 요청 수의 상한. 서버의 OLLAMA_NUM_PARALLEL과 같은 값입니다
MAX_CONCURRENCY = 32

# .batch 한 번에 보내는 입력 수. 묶음이 끝날 때마다 결과를 진행 파일에 씁니다
CHUNK_SIZE = 64

# 단계마다 결과를 이어 쓰는 진행 파일
PROGRESS_DIR = output_dir / "gemma4_progress"
PROGRESS_DIR.mkdir(exist_ok=True)
QA_PROGRESS_PATH = PROGRESS_DIR / "qa.jsonl"
QUESTIONS_PROGRESS_PATH = PROGRESS_DIR / "questions.jsonl"
ANSWER_PROGRESS_PATH = PROGRESS_DIR / "answers.jsonl"
OUT_OF_SCOPE_PROGRESS_PATH = PROGRESS_DIR / "out_of_scope.jsonl"

# 교안 02가 읽을 최종 데이터셋 파일(교안 01의 labor_qa.jsonl과 같은 키)
DATASET_PATH = output_dir / "labor_qa_gemma4.jsonl"

# 테이블의 긴 글(답, 조문)이 50자에서 잘리지 않고 끝까지 보이게 합니다
pd.set_option("display.max_colwidth", None)

print("생성용 LLM:", MODEL_NAME)

아래 세 셀은 교안 01의 코드를 그대로 옮긴 것입니다(원문 정리, 생성 대상 고르기, 숫자 대조, 금지 표현, 자카드 유사도, Q&A 생성 체인, 질문 늘리기 체인, 답 만들기 프롬프트, 범위 밖 상수). 설명은 교안 01을 봅니다.

In [ ]:
# 교안 01과 같은 함수: 원문 정리, 생성 대상 고르기, 숫자 대조, 금지 표현, 자카드 유사도
import re
import unicodedata


def clean_text(text):
    """조문을 생성용 LLM의 입력이자 숫자 대조의 기준으로 쓸 수 있게 정리합니다.

    본문이 아닌 개정 태그는 지우고, 다르게 적힌 글자와 공백은 한 가지로 맞춥니다.
    """
    # NFC 정규화: 모양은 같지만 유니코드 번호가 다른 글자를 표준 번호 하나로 바꿉니다
    # 예: 조문의 '金'은 호환용 한자(U+F90A)라서, 답에 쓴 일반 한자 '金'(U+91D1)과 문자열 비교에서 다른 글자가 됩니다
    text = unicodedata.normalize("NFC", text)

    # 가운뎃점 자리에 쓴 한글 자모(U+318D)를 가운뎃점(U+00B7)으로 바꿉니다
    # 안 바꾸면 FAQ와 생성용 LLM의 답에 쓰인 가운뎃점과 다른 글자로 셉니다
    text = text.replace("\u318d", "\u00b7")

    # <개정 2012.2.1>, <신설 2008.3.21> 같은 개정 태그를 지웁니다. 시행일자는 교안 02의 데이터 카드에 적습니다
    # 안 지우면 날짜 2012, 2, 1 때문에 '2일' 같은 틀린 답이 숫자 대조를 통과합니다
    text = re.sub(r"<[^>]*>", "", text)

    # 줄바꿈과 겹친 공백을 공백 한 칸으로 맞춥니다(split()은 둘 다 구분자로 봅니다)
    # 안 맞추면 공백 모양만 다른 같은 문장을 다른 문장으로 봅니다
    return " ".join(text.split())


# 세부 내용을 시행령(대통령령)이나 시행규칙(고용노동부령) 등에 넘기는 문장의 끝맺음
DELEGATION_ENDINGS = ("대통령령으로 정한다.", "고용노동부령으로 정한다.", "정하는 바에 따른다.")


def is_qa_target(text):
    """Q&A를 만들 문단이면 True, 위임만 하는 한 문장이거나 목적 조항이면 False를 돌려줍니다."""
    # 1) 한 문장(중간에 "다. "가 없음)이고 위임 끝맺음으로 끝나면 제외합니다
    #    endswith에 튜플을 주면 튜플 안의 끝맺음 가운데 하나로 끝나는지 봅니다
    if "다. " not in text and text.endswith(DELEGATION_ENDINGS):
        return False

    # 2) 조 제목이 (목적)인 목적 조항은 제외합니다
    #    (의\d+)?는 "제43조의2"처럼 가지 번호가 붙은 조도 맞추려는 부분입니다
    if re.match(r"^제\d+조(의\d+)?\(목적\)", text):
        return False
    return True


def numbers_match(answer, paragraph):
    """답에 나온 숫자가 모두 조문에도 있으면 True를 돌려줍니다."""
    # 조문의 숫자 리스트
    paragraph_numbers = re.findall(r"\d+", paragraph)

    # 답의 숫자 가운데 하나라도 조문에 없으면 False
    for number in re.findall(r"\d+", answer):
        if number not in paragraph_numbers:
            return False
    return True


# 프롬프트 규칙 2번에서 막은 표현과, 생성용 LLM이 일부만 답하고 붙이는 거절 문장('안내해 드릴 수 없', '제공된 조문')
BANNED_PHRASES = ["조문에 따르면", "조문상", "이 조문", "이 조항", "해당 조항", "해당 규정", "안내해 드릴 수 없", "제공된 조문"]


def has_banned_phrase(answer):
    """답에 막은 표현이 하나라도 있으면 True를 돌려줍니다."""
    for phrase in BANNED_PHRASES:
        if phrase in answer:
            return True
    return False


def jaccard(a, b):
    """공백을 뺀 두 문장의 글자 집합으로 계산한 자카드 유사도를 돌려줍니다."""
    chars_a = set(a.replace(" ", ""))
    chars_b = set(b.replace(" ", ""))
    return len(chars_a & chars_b) / len(chars_a | chars_b)

In [ ]:
# 교안 01과 같은 Q&A 스키마, 프롬프트, 체인
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class QAPair(BaseModel):
    """조문에 근거한 Q&A 한 건입니다."""
    question: str = Field(description="조문 내용만으로 답할 수 있는 직장인이나 사업주의 질문")
    answer: str = Field(description="조문에 적힌 내용으로 쓴 상담원의 한두 문장 답")
    # Literal[...]은 세 값 가운데 하나만 받는 타입입니다
    question_type: Literal["사실 확인", "절차", "문제 상황"] = Field(
        description="질문 유형. 사실 확인은 기준, 기간, 금액이 얼마인지 묻는 질문, 절차는 무엇을 언제 어떻게 해야 하는지 묻는 질문, 문제 상황은 회사가 이렇게 했는데 괜찮은지나 내가 받을 수 있는지 묻는 질문"
    )


class QASet(BaseModel):
    """조문 하나에서 만든 Q&A 리스트입니다."""
    items: list[QAPair] = Field(description="서로 다른 Q&A 리스트")


# system은 역할과 목적, human은 할 일, 목적별 규칙, 조문입니다({n}: 만들 개수의 상한, {paragraph}: 조문 본문)
# 사용자는 조문을 보지 않으므로 '조문에 따르면' 같은 말을 막습니다
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "노무 상담 챗봇이 배울 질문과 답을 써야 한다. 네가 쓴 질문과 답은 그대로 챗봇의 학습 예시가 된다."),
    ("human", """아래 [조문]으로, 직장인이나 사업주가 노무 상담 챗봇에 실제로 물어볼 Q&A를 1~{n}개 만들어라. 조문에서 물을 것이 적으면 적게 만든다.

근거:
1. [조문]에 적힌 내용만 쓰고, 조문에 없는 내용은 지어내지 않는다.

답 쓰는 법:
2. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
3. 답에는 조문에 적힌 조건(대상, 기간, 비율, 예외)을 빠뜨리지 않는다.

질문 쓰는 법:
4. 질문마다 유형(사실 확인, 절차, 문제 상황)을 정하고, 여러 개를 만들면 유형을 섞는다.
5. 같은 답이 나오는 질문은 하나만 만든다.
6. 질문은 법률 용어 대신 묻는 사람이 평소 쓰는 말로, 자기 상황을 말하며 묻는 문장으로 쓴다.

조·항 번호:
7. 질문에는 조·항 번호(제○조, 제○항)를 쓰지 않는다. 답에서 조문을 밝힐 때는 '근로기준법 제60조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다.

[조문]
{paragraph}"""),
])

# 프롬프트 -> 생성용 LLM(QASet 모양으로만 답함) 체인
# 입력 딕셔너리를 받아 QASet 객체를 돌려줍니다
qa_generator = qa_prompt | llm.with_structured_output(QASet)

In [ ]:
# 교안 01과 같은 질문 늘리기 체인, 답 만들기 프롬프트, 범위 밖 상수
class InstructionSet(BaseModel):
    """새로 만든 지시 리스트입니다."""
    items: list[str] = Field(description="서로 다른 새 지시 문장 리스트")


# 손으로 쓴 시드 지시 3개(연차휴가, 수습 해고, 배우자 출산전후휴가)
seed_instructions = [
    "입사한 지 1년이 안 됐는데 연차휴가가 생기나요?",
    "수습 기간이 끝나기 전에 회사가 그만 나오라고 하면 해고인가요?",
    "아내가 출산 예정인데 배우자 출산전후휴가는 며칠 쓸 수 있나요?",
]

# {topic}(주제 태그), {seeds}(시드 지시 줄), {n}(만들 개수)은 부를 때 채웁니다
question_prompt = ChatPromptTemplate.from_messages([
    ("system", "직장인이나 사업주가 노무 상담 챗봇에 실제로 입력할 법한 질문을 써야 한다. 네가 쓴 질문은 챗봇 학습 데이터의 질문이 된다."),
    ("human", """직장인이나 사업주가 노무 상담 챗봇에 물어볼 '{topic}' 주제의 새 지시 {n}개를 만들어라. 지시는 챗봇에 입력하는 질문이고, 아래 [시드 지시]는 그 예시다.

질문 쓰는 법:
1. 시드 지시와 표현, 상황을 다르게 쓰고, 시드 지시를 그대로 옮기거나 낱말만 바꾸지 않는다.
2. 사실 확인, 절차, 문제 상황 질문을 섞는다.
3. 묻는 사람이 자기 상황을 말하며 묻는 말투로 쓴다.

[시드 지시]
{seeds}"""),
])

# 프롬프트 -> 생성용 LLM(InstructionSet 모양으로만 답함) 체인
instruction_generator = question_prompt | llm.with_structured_output(InstructionSet)


class GroundedAnswer(BaseModel):
    """생성용 LLM이 검색으로 찾은 조문을 보고 쓴 답입니다."""
    doc_id: str = Field(description="답의 근거가 된 조문의 doc_id. 조문에 답이 없으면 '없음'")
    answer: str = Field(description="조문에 근거한 상담원의 한두 문장 답")


# {articles}에 검색으로 찾은 조문, {question}에 질문을 채웁니다
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 노동법 조문에 적힌 내용으로만 상담 질문에 답해야 한다. 네가 쓴 답은 노무 상담 챗봇이 배울 예시가 된다."),
    ("human", """아래 [질문]에 [조문]만 근거로 답하라.

근거와 답할 범위:
1. 답의 근거가 된 조문의 doc_id를 하나 적는다.
2. [조문]으로 질문의 핵심에 답할 수 있으면 조문에 적힌 내용만으로 조건을 빠뜨리지 않고 답한다. 질문 가운데 [조문]이 다루지 않는 부분은 답에서 빼고 언급하지 않는다('안내해 드릴 수 없습니다', '제공된 조문만으로는', '정해져 있지 않습니다', '판단할 수 없습니다' 같은 문장을 쓰지 않는다).
3. 자기 상황이 조문의 요건에 해당하는지 묻는 질문이면 조문의 요건을 밝히고 '이런 경우라면 ~에 해당할 수 있습니다'처럼 요건에 맞춰 답한다.
4. [조문]으로 질문의 핵심에 답할 수 없을 때만 doc_id에 "없음"을 적고, 답은 "안내해 드릴 수 없는 질문입니다."로 쓴다.

답 쓰는 법:
5. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
6. 답에서 조문을 밝힐 때는 '근로기준법 제60조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다.

[조문]
{articles}

[질문]
{question}"""),
])

# 이 챗봇이 다루는 다섯 법률로 답할 수 없는 주제와, 그 질문에 붙일 거절 답
OUT_OF_SCOPE_TOPIC = "이 챗봇이 다루는 다섯 법률(근로기준법, 최저임금법, 퇴직급여법, 남녀고용평등법, 기간제법)로는 답할 수 없는 질문(연말정산 같은 세금, 실업급여와 고용보험, 4대보험 가입, 이직 준비, 개인 투자)"
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 요청할 범위 밖 질문 수: 교안 02의 증강과 필터 뒤에도 범위 밖 행이 3~5%가 되게 하는 개수입니다
OUT_OF_SCOPE_COUNT = 70

## 3. 본실습 데이터셋 만들기

교안 01과 같은 단계(원문 정리부터 저장까지)로 본실습 데이터셋을 Gemma 4 E4B로 만듭니다. 로컬 LLM은 수백 건 생성에 몇 분에서 수십 분이 걸려, 그 사이 커널 재시작, Ollama 서버의 GPU 메모리 부족, 연결 끊김으로 멈추거나 일부 요청이 실패하기 쉽습니다. 그래서 단계마다 결과를 진행 파일에 이어 쓰고, 다시 실행하면 끝난 입력은 건너뛰고 실패한 입력만 다시 보냅니다.

| 단계 | 모델 호출 | 진행 파일의 키 | 진행 파일(`output/gemma4_progress/`) |
|---|---|---|---|
| 조문 읽기, 정리, 생성 대상 고르기 | 없음 | - | - |
| Q&A 생성 | 생성 대상 문단마다 1회 | 문단의 `doc_id` | `qa.jsonl` |
| 근거 검수와 표본 검토 | 없음 | - | - |
| 분포 확인 | 없음 | - | - |
| 질문 늘리기 | 주제 태그마다 1회 | 주제 태그 | `questions.jsonl` |
| 새 질문에 답 만들기 | 남은 지시마다 1회 | 남은 지시의 질문 문장 | `answers.jsonl` |
| 범위 밖 거절 행 | 1회 | `out_of_scope` | `out_of_scope.jsonl` |
| 저장 | 없음 | - | - |

### 3-1. 이어서 처리하는 함수

교안 01 함께 따라하기의 진행 파일, `CHUNK_SIZE`, `"a"` 모드 이어 쓰기와 같은 구조입니다. 이 부록은 같은 구조를 모델을 부르는 네 단계에서 쓰므로 함수 두 개로 묶었습니다.

- **진행 파일 한 줄:** `{"key": 키, "result": 결과}`입니다. 결과는 구조화 출력 객체를 `.model_dump()`로 바꾼 딕셔너리입니다.
- **`load_progress(path)`:** 진행 파일을 읽어 키 -> 결과 딕셔너리를 돌려줍니다. 파일이 없으면(처음 실행) 빈 딕셔너리입니다.
- **`run_with_progress(chain, inputs, keys, path, chunk_size, max_concurrency)`:** `inputs`와 같은 순서의 `keys` 가운데 진행 파일에 없는 키의 입력만 `chunk_size`개씩 `.batch(..., return_exceptions=True)`로 보냅니다. 묶음이 끝날 때마다 성공한 결과를 진행 파일 끝에 이어 쓰고, 이번 실행에서 실패한 키 리스트를 돌려줍니다.
- **실패한 입력:** 진행 파일에 쓰지 않으므로, 같은 셀을 다시 실행하면 그 입력만 다시 보냅니다. 묶음마다 파일을 열고 닫으므로, 커널이 멈춰도 이미 끝난 묶음의 결과는 파일에 남습니다.

In [ ]:
def load_progress(path):
    """진행 파일을 읽어 키 -> 결과 딕셔너리를 돌려줍니다(파일이 없으면 빈 딕셔너리)."""
    done = {}

    # 처음 실행해 진행 파일이 아직 없으면 빈 딕셔너리를 돌려줍니다
    if not path.exists():
        return done

    # 한 줄이 {"key": 키, "result": 결과 딕셔너리} 하나이고, 빈 줄은 건너뜁니다
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip() != "":
                record = json.loads(line)
                done[record["key"]] = record["result"]
    return done


def run_with_progress(chain, inputs, keys, path, chunk_size, max_concurrency):
    """진행 파일에 없는 키의 입력만 묶음으로 보내 결과를 이어 쓰고, 이번 실행에서 실패한 키 리스트를 돌려줍니다."""
    # 진행 파일에 결과가 있는 키는 끝난 입력입니다
    done = load_progress(path)

    # 끝나지 않은 입력과 그 키만 같은 순서로 모읍니다
    pending_inputs = []
    pending_keys = []
    for item, key in zip(inputs, keys):
        if key not in done:
            pending_inputs.append(item)
            pending_keys.append(key)
    print("보낼 입력 수:", len(pending_inputs), "/", len(inputs))

    # 남은 입력을 chunk_size개씩 잘라 보내고, 실패해도 나머지 결과를 받습니다
    failed_keys = []
    start = time.perf_counter()
    for chunk_start in range(0, len(pending_inputs), chunk_size):
        chunk_inputs = pending_inputs[chunk_start:chunk_start + chunk_size]
        chunk_keys = pending_keys[chunk_start:chunk_start + chunk_size]
        chunk_results = chain.batch(chunk_inputs, config={"max_concurrency": max_concurrency}, return_exceptions=True)

        # 묶음이 끝나면 진행 파일을 이어 쓰기 모드로 열고, 성공한 결과만 키와 함께 한 줄씩 씁니다
        chunk_failed_count = 0
        with open(path, "a", encoding="utf-8") as file:
            for key, result in zip(chunk_keys, chunk_results):
                if isinstance(result, Exception):
                    failed_keys.append(key)
                    chunk_failed_count += 1

                    # 묶음의 첫 실패만 내용을 찍어 원인(연결 오류, 출력 모양 오류)을 봅니다
                    if chunk_failed_count == 1:
                        print("실패 예:", key, repr(result)[:200])
                else:
                    record = {"key": key, "result": result.model_dump()}
                    file.write(f"{json.dumps(record, ensure_ascii=False)}\n")
        # 번호는 이번 실행에서 보낼 입력(pending_inputs) 안의 순번이라, 다시 실행하면 1부터 다시 셉니다
        print("이번 실행 묶음:", chunk_start + 1, "~", chunk_start + len(chunk_inputs), "| 실패:", chunk_failed_count, "| 경과 시간(초):", round(time.perf_counter() - start))

    # 다시 실행하면 이 키의 입력만 다시 보냅니다
    print("이번 실패 수:", len(failed_keys))
    return failed_keys

### 3-2. 조문 읽기, 정리, 생성 대상 고르기

교안 01과 같이 노동법 408문단을 읽어 본문을 정리하고, Q&A를 만들 생성 대상 382문단을 고릅니다. 모델을 부르지 않는 단계라 진행 파일이 없습니다.

In [ ]:
# 노동법 조문 문단: 문단마다 doc_id, law, source(주제 태그), text(원문)
law_articles = json.loads((data_dir / "본실습" / "labor_law_articles.json").read_text(encoding="utf-8"))

# 주제 태그별 문단 수. 질문 늘리기에서 주제 태그의 순서로도 씁니다
source_counts = pd.DataFrame(law_articles)["source"].value_counts()

# 전체 문단의 본문을 교안 01의 원문 정리 함수로 정리합니다
for article in law_articles:
    article["text"] = clean_text(article["text"])

# 목적 조항과 위임 문장 하나뿐인 문단을 빼고 생성 대상 문단만 모읍니다
qa_articles = []
for article in law_articles:
    if is_qa_target(article["text"]):
        qa_articles.append(article)

print("문단 수:", len(law_articles), "| 생성 대상 문단 수:", len(qa_articles))

### 3-3. Q&A 생성

교안 01과 같은 입력(`{"n": 3, "paragraph": 본문}`)을 생성 대상 382문단에 보냅니다. 진행 파일의 키는 문단의 `doc_id`입니다.

- **걸리는 시간:** 이번 비교 측정의 처리량(동시 32에서 초당 1.103개)으로 어림하면 약 6분입니다.
- **실패:** 셀이 끝나면 `이번 실패 수`를 봅니다. 0이 아니면 같은 셀을 다시 실행해 실패한 문단만 다시 보냅니다.

In [ ]:
# 생성 대상 문단마다 체인 입력과 키(doc_id)를 같은 순서로 만듭니다
qa_inputs = [{"n": 3, "paragraph": article["text"]} for article in qa_articles]
qa_keys = [article["doc_id"] for article in qa_articles]

# 진행 파일에 없는 문단만 보내고, 묶음이 끝날 때마다 결과를 이어 씁니다
qa_failed_keys = run_with_progress(qa_generator, qa_inputs, qa_keys, QA_PROGRESS_PATH, CHUNK_SIZE, MAX_CONCURRENCY)

진행 파일 전체를 읽어 Q&A 행으로 펼칩니다. 여러 번 나눠 실행한 결과도 한 파일에 이어 쓰여 있습니다.

- **딕셔너리로 읽기:** 진행 파일의 결과는 `QASet` 객체가 아니라 딕셔너리이므로 `item["question"]`처럼 키로 읽습니다.
- **id:** 생성 대상 문단 순서대로 읽으며 `qa-0001`부터 붙입니다. 진행 파일에 쓰인 순서와 상관없이 같은 결과면 같은 id가 붙습니다.

In [ ]:
# 진행 파일을 읽어 doc_id -> 결과 딕셔너리를 만듭니다
qa_done = load_progress(QA_PROGRESS_PATH)

# 생성 대상 문단 순서대로 결과를 행으로 펼치고, 아직 결과가 없는 문단은 건너뜁니다
qa_rows = []
for article in qa_articles:
    if article["doc_id"] in qa_done:
        for item in qa_done[article["doc_id"]]["items"]:
            qa_rows.append({
                # :04d 는 네 자리가 되도록 앞을 0으로 채웁니다(1 -> 0001)
                "id": f"qa-{len(qa_rows) + 1:04d}",
                "doc_id": article["doc_id"],
                "source": article["source"],
                "method": "문서 생성",
                "question": item["question"],
                "answer": item["answer"],
                "question_type": item["question_type"],
            })

print("결과가 있는 문단 수:", len(qa_done), "/", len(qa_articles))
print("Q&A 행 수:", len(qa_rows))
qa_table = pd.DataFrame(qa_rows)
display(qa_table.head())

### 3-4. 근거 검수와 표본 검토

교안 01의 근거 검수와 같이 답의 숫자를 근거 조문과 대조하고 금지 표현을 찾아, 둘 다 통과한 행만 `passed_rows`에 남깁니다.

In [ ]:
# doc_id로 정리한 본문을 바로 찾도록 딕셔너리를 만듭니다
doc_texts = {article["doc_id"]: article["text"] for article in law_articles}

# 행마다 숫자 대조와 금지 표현 결과를 열로 붙이고, 두 열의 조합별 건수를 셉니다
qa_table["숫자 대조"] = [numbers_match(row["answer"], doc_texts[row["doc_id"]]) for row in qa_rows]
qa_table["금지 표현"] = [has_banned_phrase(row["answer"]) for row in qa_rows]
display(qa_table[["숫자 대조", "금지 표현"]].value_counts())

# 숫자 대조를 통과하고 금지 표현이 없는 행만 남깁니다
passed_rows = []
for row in qa_rows:
    if numbers_match(row["answer"], doc_texts[row["doc_id"]]) and not has_banned_phrase(row["answer"]):
        passed_rows.append(row)
print("근거 검수 통과:", len(passed_rows), "/", len(qa_rows))

앞의 품질 판정에서 Gemma 4 E4B의 오류(조건·예외 누락 7.0%, 할루시네이션 1.4%)는 숫자 대조와 금지 표현 검사를 모두 통과했습니다. 그래서 로컬 LLM으로 만든 데이터는 통과한 행에서 표본 20행을 뽑아 근거 조문과 나란히 읽습니다.

- **`.sample(n=sample_size, random_state=42)`:** 데이터프레임에서 `sample_size`행을 무작위로 뽑습니다. `random_state`(시드)를 고정하면 같은 행 리스트에서 늘 같은 행이 뽑힙니다.
- **`sample_size`:** 20으로 두고, 통과한 행이 20개보다 적으면 그 행 수로 줄여 모두 봅니다. `.sample`은 행 수보다 큰 `n`을 받으면 오류를 냅니다.

In [ ]:
# 표본 크기는 20행이고, 통과한 행이 20개보다 적으면 통과한 행 수로 줄입니다
sample_size = 20
if len(passed_rows) < sample_size:
    sample_size = len(passed_rows)

# 통과한 행에서 표본을 시드를 고정해 뽑습니다
review_table = pd.DataFrame(passed_rows).sample(n=sample_size, random_state=42)

# 표본마다 근거 조문을 옆에 붙여 사람이 대조합니다
review_table["근거 조문"] = [doc_texts[doc_id] for doc_id in review_table["doc_id"]]
display(review_table[["id", "question", "answer", "근거 조문"]])

**결과 읽기**
- **읽을 것:** 조문의 조건과 예외("다만" 뒤 등)가 답에 남았는지, 결론이 조문과 같은지 봅니다.
- **판단:** 품질 판정에서 Gemma 4 E4B는 정확이 아닌 답이 열에 하나꼴(11.3%)이었으므로, 20행 가운데 틀린 행이 이보다 크게 많으면 전체를 읽거나 이 단계를 API로 다시 만듭니다.

### 3-5. 분포 확인

교안 01과 같이 주제별 행 수와 질문 유형별 행 수를 봅니다. 주제별 문서 생성 행 수는 다음 질문 늘리기 단계에서 주제마다 요청할 질문 개수를 정합니다.

In [ ]:
# 주제 태그별 생성 대상 문단 수와 Q&A 행 수를 나란히 놓습니다
qa_source_counts = pd.DataFrame(qa_articles)["source"].value_counts()
topic_coverage = pd.DataFrame({"문단 수": qa_source_counts, "Q&A 행 수": qa_table["source"].value_counts()})
display(topic_coverage)

In [ ]:
# 질문 유형별 행 수
display(qa_table["question_type"].value_counts())

**결과 읽기**
- **질문 유형:** 품질 판정에서 Gemma 4 E4B는 71건 가운데 58건(81.7%)이 사실 확인이었습니다. 이번에도 사실 확인이 대부분이면 질문 늘리기의 하한 `MIN_NEW_PER_TOPIC`을 늘려 다른 유형의 질문을 채웁니다.

### 3-6. 질문 늘리기

교안 01과 같이 주제 태그마다 새 지시를 요청합니다. 주제별 문서 생성 행 수의 중앙값까지 모자란 만큼 요청하되 `MIN_NEW_PER_TOPIC`(10)개 이상 `MAX_NEW_PER_TOPIC`(30)개 이하로 맞추고, 진행 파일의 키는 주제 태그입니다.

- **실행 순서:** Q&A 생성의 실패 수가 0이 된 뒤에 실행합니다. 주제별 Q&A 행 수가 요청 개수를 정하기 때문입니다.
- **요청 개수를 바꿀 때:** 끝난 주제는 건너뛰므로 진행 파일을 두면 바뀐 개수가 적용되지 않습니다. `output/gemma4_progress/questions.jsonl`을 지운 뒤 실행합니다.

In [ ]:
# 시드 지시를 "- 지시" 줄로 잇습니다(모든 주제에 같은 시드)
seed_lines = "\n".join([f"- {seed}" for seed in seed_instructions])

# 주제별 요청 개수: 문서 생성 행 수가 중앙값(목표)에 모자란 만큼 요청하되, 10개 이상 30개 이하로
MIN_NEW_PER_TOPIC = 10
MAX_NEW_PER_TOPIC = 30

# 주제별 문서 생성 행 수와 그 중앙값
topic_row_counts = qa_table["source"].value_counts()
median_rows = topic_row_counts.median()

# 주제 태그 12개마다 입력 딕셔너리를 만듭니다. 키는 주제 태그입니다
topics = list(source_counts.index)
instruction_inputs = []
for topic in topics:
    # 1) 목표(중앙값)까지 모자란 행 수
    n = int(median_rows - topic_row_counts.get(topic, 0))

    # 2) 큰 주제도 자기 상황을 말하는 질문을 조금은 섞도록 하한, 문단이 적은 주제에 비슷한 질문이 몰리지 않도록 상한
    if n < MIN_NEW_PER_TOPIC:
        n = MIN_NEW_PER_TOPIC
    if n > MAX_NEW_PER_TOPIC:
        n = MAX_NEW_PER_TOPIC
    instruction_inputs.append({"topic": topic, "seeds": seed_lines, "n": n})

# 진행 파일에 없는 주제만 보내고, 묶음이 끝날 때마다 결과를 이어 씁니다
instruction_failed_keys = run_with_progress(instruction_generator, instruction_inputs, topics, QUESTIONS_PROGRESS_PATH, CHUNK_SIZE, MAX_CONCURRENCY)

In [ ]:
# 진행 파일을 읽어 주제 태그 순서대로 새 지시를 모읍니다
instruction_done = load_progress(QUESTIONS_PROGRESS_PATH)
new_instructions = []
for topic in topics:
    if topic in instruction_done:
        for instruction in instruction_done[topic]["items"]:
            new_instructions.append({"source": topic, "question": instruction})

print("결과가 있는 주제 수:", len(instruction_done), "/", len(topics))
print("새 지시 수:", len(new_instructions))
display(pd.DataFrame(new_instructions).head())

In [ ]:
# 비교 대상: 시드 지시 + 지금까지 남긴 지시(교안 01과 같은 자카드 0.8 기준)
compared_texts = list(seed_instructions)

kept_instructions = []
for instruction in new_instructions:
    # 비교 대상 가운데 하나라도 0.8 이상이면 비슷한 지시로 보고, 더 비교하지 않고 멈춥니다
    is_similar = False
    for other in compared_texts:
        if jaccard(instruction["question"], other) >= 0.8:
            is_similar = True
            break

    # 비슷하지 않은 지시만 남기고, 다음 지시의 비교 대상에 더합니다
    if not is_similar:
        kept_instructions.append(instruction)
        compared_texts.append(instruction["question"])

print("남은 지시 수:", len(kept_instructions), "/", len(new_instructions))

### 3-7. 새 질문에 답 만들기

교안 01과 같은 코드로 단원 폴더의 검색기 파일(`law_retriever.py`)을 불러 408문단 전체의 검색기를 만들고, 남은 지시마다 하이브리드 검색으로 찾은 조문의 앞 5개(RRF 순서)를 넣어 생성용 LLM이 근거 doc_id와 답을 씁니다. 검색기의 구성과 설정 이유는 교안 01을 봅니다.

- **임베딩은 OpenAI:** 검색기는 교안 01과 같은 OpenAI 임베딩 모델(`text-embedding-3-large`)을 쓰므로, 아래 첫 셀이 단원 폴더 `.env`의 `OPENAI_API_KEY`를 읽습니다. 조문과 질문은 OpenAI로 보내지므로, 반출이 금지된 자료라면 임베딩도 로컬 모델로 바꿔야 회사 밖으로 나가는 것이 없습니다.
- **재정렬은 끔:** 교안 01과 같이 `use_rerank=False`로 재정렬을 끄고 리랭커 모델을 불러오지 않습니다. 파드의 GPU로는 `use_rerank=True`로 재정렬을 켤 수도 있지만, 이 부록은 교안 01과 같이 끈 채로 둡니다.
- **서버 설정은 그대로:** 요청마다 조문 5개만 들어가고, 가장 긴 다섯 문단을 합쳐도 3,902자입니다. 이번 비교 측정의 글자당 토큰 비율(약 6,200토큰 / 9,200자)로 어림하면 약 2,600토큰(측정하지 않은 값)이라, 규칙 문장과 답을 더해도 생성용 서버의 문맥 4096 안에 들어갑니다. 서버 터미널에 입력을 잘랐다는 경고(`truncating input prompt`)가 나오면 서버를 `OLLAMA_CONTEXT_LENGTH=8192`로 다시 띄우고 체인의 `llm`에 `num_ctx=8192`를 줍니다.

In [ ]:
import sys

from dotenv import find_dotenv, load_dotenv

# 검색기의 임베딩 모델이 읽을 OPENAI_API_KEY를 단원 폴더 .env에서 읽습니다
load_dotenv(find_dotenv(usecwd=True))

# 단원 폴더의 검색기 파일(law_retriever.py)을 import 할 수 있게 경로에 더합니다
sys.path.append(str(material_dir))
from law_retriever import build_retriever

# 408문단 전체로 검색기를 만듭니다. 생성 대상에서 뺀 위임 문단과 목적 조항도 다른 조문과 함께 근거가 될 수 있어 넣습니다
# 정리한 본문을 여기서 한 번 임베딩해 Chroma에 넣습니다. 키워드 검색과 의미 검색이 5개씩 찾아 합칩니다(최대 10개)
# 수업 시간 관계상 재정렬은 일부러 끕니다(use_rerank=True로 켜면 순서가 더 정확하지만 GPU 없이는 느립니다)
article_retriever = build_retriever(law_articles, collection_name="labor_law", candidate_k=5, use_rerank=False)
print("검색 대상 문단 수:", len(law_articles))

In [ ]:
# 남은 지시 하나로 검색 결과를 봅니다(질문 임베딩 1회)
sample_question = kept_instructions[0]["question"]
print("질문:", sample_question)
print("질문의 주제 태그:", kept_instructions[0]["source"])
sample_documents = article_retriever.invoke(sample_question)[:5]  # 합친 후보(최대 10개)에서 RRF 순서의 앞 5개만 씁니다

# 검색한 조문을 RRF 순서대로 테이블로 봅니다
retrieved_rows = []
for document in sample_documents:
    retrieved_rows.append({"doc_id": document.metadata["doc_id"], "source": document.metadata["source"], "text": document.page_content})
display(pd.DataFrame(retrieved_rows))

In [ ]:
# 남은 지시마다 조문을 검색하고, 검색한 조문을 "doc_id: 본문" 줄로 이어 체인 입력을 만듭니다
# 질문마다 검색한 doc_id 리스트는 뒤에서 생성용 LLM이 고른 근거를 확인할 때 씁니다
retrieved_doc_ids = []
answer_inputs = []
for index, instruction in enumerate(kept_instructions, start=1):
    # 1) 하이브리드 검색으로 조문 찾기(질문 임베딩 1회)
    documents = article_retriever.invoke(instruction["question"])[:5]  # RRF 순서의 앞 5개만 씁니다
    if index % 50 == 0:  # 질문마다 임베딩 요청을 보내므로 50개마다 진행 수를 찍습니다
        print("검색한 지시 수:", index, "/", len(kept_instructions))

    # 2) 생성용 LLM이 고를 doc_id를 줄 앞에 두고, 줄바꿈으로 잇기
    lines = []
    doc_ids = []
    for document in documents:
        lines.append(f"{document.metadata['doc_id']}: {document.page_content}")
        doc_ids.append(document.metadata["doc_id"])
    retrieved_doc_ids.append(doc_ids)
    answer_inputs.append({"articles": "\n".join(lines), "question": instruction["question"]})

# 보낼 입력 하나를 봅니다: 질문, 검색한 doc_id, {articles}에 들어갈 조문 줄
print("질문:", answer_inputs[0]["question"])
print("검색한 doc_id:", retrieved_doc_ids[0])
print(answer_inputs[0]["articles"])

- **키는 질문 문장:** 남은 지시의 질문 문장을 그대로 키로 씁니다. 비슷한 지시 거르기(자카드 0.8)가 같은 문장을 이미 지웠으므로 키가 겹치지 않습니다. 질문 늘리기의 실패 수가 0이 된 뒤에 실행합니다.
- **질문 늘리기가 바뀌었을 때:** 실패했던 주제를 다시 보내 지시가 늘거나 질문 늘리기를 다시 만들어도, 이미 답을 받은 질문은 그 답을 그대로 쓰고 새 질문만 보냅니다. 순번을 키로 쓰면 지시가 중간에 끼어들 때 뒤 지시의 순번이 밀려, 다른 질문의 답이 붙습니다.
- **다시 실행할 때:** 검색은 진행 파일과 관계없이 남은 지시 전체를 다시 하고, 생성용 LLM에는 진행 파일에 없는 질문만 보냅니다.

In [ ]:
# 답 만들기 체인: 교안 01과 같은 프롬프트와 구조화 출력, 생성용 서버 설정(문맥 4096, 동시 32) 그대로
answer_generator = answer_prompt | llm.with_structured_output(GroundedAnswer)

# 진행 파일의 키: 남은 지시의 질문 문장(answer_inputs와 같은 순서)
answer_keys = []
for instruction in kept_instructions:
    answer_keys.append(instruction["question"])

# 진행 파일에 없는 지시만 보내고, 묶음이 끝날 때마다 결과를 이어 씁니다
answer_failed_keys = run_with_progress(answer_generator, answer_inputs, answer_keys, ANSWER_PROGRESS_PATH, CHUNK_SIZE, MAX_CONCURRENCY)

교안 01과 같이 생성용 LLM이 고른 `doc_id`가 그 질문에서 검색한 조문 가운데 하나면 `answer_rows`에, "없음"이거나 검색한 조문에 없는 id면 `no_source_rows`에 나눕니다. 근거가 있어도 답에 거절 문장 같은 막은 표현이 있으면 `banned_rows`로 따로 뺍니다.

In [ ]:
# 진행 파일을 읽어 키 -> 생성용 LLM의 답(doc_id, answer) 딕셔너리를 만듭니다
answer_done = load_progress(ANSWER_PROGRESS_PATH)

# 남은 지시 순서대로 결과를 나누고, 아직 결과가 없는 지시는 건너뜁니다
answer_rows = []
no_source_rows = []
banned_rows = []
for key, instruction, question_doc_ids in zip(answer_keys, kept_instructions, retrieved_doc_ids):
    if key in answer_done:
        result = answer_done[key]

        # 근거 doc_id가 그 질문에서 검색한 조문이 아니면("없음" 포함) 근거 없는 행입니다
        if result["doc_id"] not in question_doc_ids:
            no_source_rows.append({"source": instruction["source"], "question": instruction["question"], "doc_id": result["doc_id"], "answer": result["answer"]})
        # 근거는 있지만 답에 막은 표현(일부만 답하고 붙인 거절 문장 포함)이 있으면 따로 모읍니다
        elif has_banned_phrase(result["answer"]):
            banned_rows.append({"source": instruction["source"], "question": instruction["question"], "doc_id": result["doc_id"], "answer": result["answer"]})
        else:
            answer_rows.append({
                "id": f"nq-{len(answer_rows) + 1:04d}",
                "doc_id": result["doc_id"],
                "source": instruction["source"],
                "method": "질문 늘리기",
                "question": instruction["question"],
                "answer": result["answer"],
            })

print("결과가 있는 지시 수:", len(answer_rows) + len(no_source_rows) + len(banned_rows), "/", len(kept_instructions))
print("근거 있는 행 수:", len(answer_rows))
print("근거 없는 행 수:", len(no_source_rows))
print("막은 표현이 있어 뺀 행 수:", len(banned_rows))

# 근거 없는 행(열 이름을 정해 두면 0건이어도 빈 테이블이 만들어집니다)
no_source_table = pd.DataFrame(no_source_rows, columns=["source", "question", "doc_id", "answer"])
display(no_source_table.head(10))

In [ ]:
# 근거 있는 행도 숫자 대조를 통과하고 금지 표현이 없는 행만 남깁니다
answer_passed_rows = []
for row in answer_rows:
    if numbers_match(row["answer"], doc_texts[row["doc_id"]]) and not has_banned_phrase(row["answer"]):
        answer_passed_rows.append(row)
print("근거 검수 통과:", len(answer_passed_rows), "/", len(answer_rows))
display(pd.DataFrame(answer_passed_rows).head())

**결과 읽기**
- **근거 없는 행:** 교안 01과 같이 학습 데이터에서 뺍니다. Gemma 4 E4B는 품질 판정에서 근거 조문이 입력에 있는 질문 1건을 "없음"으로 거절했으므로, 근거 없는 행의 질문도 몇 개 읽습니다.
- **사람이 읽을 것:** 근거 있는 행도 몇 개를 근거 조문(`doc_texts[doc_id]`)과 대조합니다.

### 3-8. 범위 밖 거절 행

교안 01과 같이 `instruction_generator`로 범위 밖 질문 70개(`OUT_OF_SCOPE_COUNT`)를 요청하고(호출 1회), 스키마가 개수를 강제하지 않으므로 앞에서부터 70개까지만 고정 거절 답 `REFUSAL_ANSWER`를 붙입니다. 요청 하나도 `run_with_progress`로 보내면 결과가 진행 파일(키 `out_of_scope`)에 남아, 다시 실행할 때 다시 보내지 않습니다.

In [ ]:
# 범위 밖 주제로 질문 OUT_OF_SCOPE_COUNT개를 요청하는 입력 하나와 그 키
out_of_scope_inputs = [{"topic": OUT_OF_SCOPE_TOPIC, "seeds": seed_lines, "n": OUT_OF_SCOPE_COUNT}]
out_of_scope_keys = ["out_of_scope"]

# 진행 파일에 결과가 있으면 보내지 않습니다
out_of_scope_failed_keys = run_with_progress(instruction_generator, out_of_scope_inputs, out_of_scope_keys, OUT_OF_SCOPE_PROGRESS_PATH, CHUNK_SIZE, MAX_CONCURRENCY)

In [ ]:
# 진행 파일을 읽어 질문마다 거절 답을 붙인 행을 만듭니다
out_of_scope_done = load_progress(OUT_OF_SCOPE_PROGRESS_PATH)
refusal_rows = []
if "out_of_scope" in out_of_scope_done:
    # 스키마가 개수를 강제하지 않아 더 많이 올 수 있으므로, 앞에서부터 OUT_OF_SCOPE_COUNT개만 씁니다
    for question in out_of_scope_done["out_of_scope"]["items"][:OUT_OF_SCOPE_COUNT]:
        refusal_rows.append({
            "id": f"out-{len(refusal_rows) + 1:04d}",
            "doc_id": "없음",
            "source": "범위 밖",
            "method": "범위 밖",
            "question": question,
            "answer": REFUSAL_ANSWER,
        })

# 사람이 질문을 모두 읽을 수 있게 id와 질문을 테이블로 봅니다(열 이름을 정해 두면 0건이어도 빈 테이블이 만들어집니다)
# 테이블이 60행을 넘으면 앞뒤 몇 행만 보이므로, 보여 줄 최대 행 수를 200으로 늘립니다
pd.set_option("display.max_rows", 200)
refusal_table = pd.DataFrame(refusal_rows, columns=["id", "doc_id", "source", "method", "question", "answer"])
print("거절 행 수:", len(refusal_rows))
display(refusal_table[["id", "question"]])

**결과 읽기**
- **사람이 읽을 것:** 교안 01과 같이 다섯 법률로 답할 수 있는 질문(예: 퇴직금, 연차)이 섞였는지 읽고, 섞였으면 그 행을 `refusal_rows`에서 뺍니다.

### 3-9. 저장

교안 01의 저장과 같은 순서와 키로 씁니다. 근거 검수를 통과한 문서 생성 행과 질문 늘리기 행에 거절 행을 더해 `output/labor_qa_gemma4.jsonl`에 저장합니다.

In [ ]:
# 근거 검수를 통과한 문서 생성 행, 질문 늘리기 행에 거절 행을 더합니다
dataset_rows = passed_rows + answer_passed_rows + refusal_rows

# 한 줄에 행 하나씩 JSON으로 씁니다. "w"는 파일을 비우고 새로 쓰므로 다시 실행해도 행이 겹치지 않습니다
with open(DATASET_PATH, "w", encoding="utf-8") as file:
    for row in dataset_rows:
        file.write(f"{json.dumps(row, ensure_ascii=False)}\n")
print("저장한 파일:", DATASET_PATH, "| 행 수:", len(dataset_rows))

# 방법별 저장 행 수
dataset_table = pd.DataFrame(dataset_rows)
display(dataset_table["method"].value_counts())

- **키:** 교안 01의 `labor_qa.jsonl`과 같습니다(`id`, `doc_id`, `source`, `method`, `question`, `answer`, 문서 생성 행은 `question_type`도).
- **진행 파일과 다른 점:** 진행 파일은 단계마다 생성용 LLM의 결과를 그대로 모은 파일이고, 이 파일은 근거 검수를 거친 학습 데이터입니다.

## 4. 교안 02에서 쓰기

교안 02에서 교안 01 데이터를 불러오는 셀의 파일 이름만 바꾸면 증강부터 그대로 이어집니다.

```python
qa_rows = load_jsonl(output_dir / "labor_qa_gemma4.jsonl")
```

- **맥에서 교안 02를 실행할 때:** 파드의 `output/labor_qa_gemma4.jsonl`을 내려받아 맥의 같은 단원 `output/` 폴더에 둡니다.
- **증강도 로컬 LLM으로:** 교안 02 맨 앞 준비 셀의 `model_name`과 `llm` 두 줄을 아래로 바꾸면 나머지 체인과 `.batch`는 그대로입니다. 서버는 생성용 설정(동시 32, 문맥 4096) 그대로 씁니다.

```python
from langchain_ollama import ChatOllama

model_name = "gemma4:e4b-it-q4_K_M"
llm = ChatOllama(model=model_name, reasoning=False)
```

- **걸리는 시간:** 증강 호출은 약 1,100회(패러프레이즈 약 700회 + 역번역 400회)입니다. 이번 비교 측정의 패러프레이즈 처리량(동시 32에서 초당 3.996개)으로 어림하면 약 5분(1,100 / 3.996 = 275초)입니다.
- **품질:** 앞의 품질 판정에서 Gemma 4 E4B의 패러프레이즈는 39개 가운데 38개(97.4%)가 원문의 뜻을 유지했습니다.
- **데이터 카드와 저장 파일:** 데이터 카드 출처 줄의 생성용 LLM은 교안 02 준비 셀의 `model_name`으로 채워지므로, 증강을 API로 했다면 카드 f-string의 `{model_name}` 자리에 `gemma4:e4b-it-q4_K_M`을 적습니다. 저장 셀의 `labor_train.jsonl`, `labor_val.jsonl`, `labor_dataset_card.md`는 `labor_train_gemma4.jsonl`처럼 다른 이름으로 바꿔, API로 만든 파일을 덮어쓰지 않게 합니다.

## 5. 핵심 정리

- **측정 방법:** API와 로컬 LLM 다섯 개를 같은 표본(Q&A 생성 40문단, 패러프레이즈 40개, 답 만들기 요청 3건)으로 한 번 측정했습니다. 품질은 이름을 A~F로 가린 LLM 검토자 한 명이 Q&A마다 근거 문단과 대조해 판정했습니다.
- **지표:** 정확, 할루시네이션(조문에 없는 내용 추가 + 숫자·조건 바뀜), 조건·예외 누락은 판정 건수를 그 생성용 LLM이 만든 Q&A 전체 건수(64~116건)로 나눈 비율입니다.
- **결과:** `gpt-5.6-luna`는 정확 96.2%, 교안 1회 약 2.8분, 약 0.64달러이고, 로컬 후보 Gemma 4 E4B는 정확 88.7%, 약 19분, 약 0.19달러입니다. Qwen3.5는 할루시네이션이 25%를 넘어 쓰지 않습니다.
- **필터의 한계:** 숫자 대조와 금지 표현 검사는 Gemma 4의 오류를 하나도 걸러 내지 못했습니다. 로컬 LLM으로 만든 데이터는 표본을 근거 조문과 나란히 읽어 확인합니다.
- **동시 처리:** `.batch`의 `max_concurrency`는 보내는 요청 수만 정하고 실제 처리량은 서버가 정합니다. Gemma 4 E4B는 측정한 값(1, 8, 16, 32) 가운데 `OLLAMA_NUM_PARALLEL=32`에서 가장 빨랐습니다. 새 질문에 답 만들기도 검색한 조문 5개만 넣으므로 같은 서버 설정(문맥 4096)으로 보냅니다.
- **이어서 처리:** `run_with_progress`가 묶음마다 성공한 결과를 진행 파일에 `"a"` 모드로 이어 쓰고, 다시 실행하면 끝난 키를 건너뛰어 실패하거나 멈춘 입력만 보냅니다. 행과 id는 결과 셀이 진행 파일을 입력 순서대로 읽을 때 만듭니다.
- **고르는 기준:** 회사 밖으로 보낼 수 없는 자료면 로컬 LLM을, 그렇지 않으면 품질과 시간에서 앞선 API를 쓰고, 어느 쪽이든 출력의 라이선스나 이용 약관을 확인합니다.